## Range and Composite Indexes in Azure Cosmos DB

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv

### Setting up the Environment

In [1]:
import os 
from dotenv import load_dotenv
import json

load_dotenv()

endpoint = os.getenv("COSMOSDB_ENDPOINT")
key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME")

### Creating the CosmosDB Client

In [2]:
from azure.cosmos import CosmosClient

client = CosmosClient(endpoint, key)

### Navigate the Resource Hierarchy

In [3]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Create a Query Statistics Helper Function

In [4]:
def run_query(query):

    items = list(
        container.query_items(
            query=query,
            enable_cross_partition_query=True
        )
    )

    headers = container.client_connection.last_response_headers

    print("\n=== Query Statistics ===")
    print(f"Documents Returned : {len(items)}")
    print(f"RU Charge          : {headers.get('x-ms-request-charge')}")
    print(f"Duration (ms)      : {headers.get('x-ms-request-duration-ms')}")
    print("Query Results:")
    for item in items:
        print(f"  - {item}")

    return items

### Create a Range Index on the Rating Property

In [5]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")


=== Indexing Policy Updated ===


### Run a Range-Filter Query After Index Optimization

In [6]:
query = """
SELECT *
FROM c
WHERE c.rating > 4.5
"""

results = run_query(query)


=== Query Statistics ===
Documents Returned : 57
RU Charge          : 4.74
Duration (ms)      : 21.777
Query Results:
  - {'category': 'Smoothies', 'name': 'Ashunti`Way Smoothie', 'description': 'Fruit n greens, mango bananas, tropical fruit blend, dragon fruit mix, mango, bananas, pineapples, apples, and spinach. Special green with strawberry bananas juice blend . Our fruity tasty smoothies are blended to perfection.', 'price': '5.49 USD', 'id': 'food0001', 'restaurantId': 'rest-2', 'rating': 4.6, 'reviewCount': 262, 'priceValue': 5.49, 'currency': 'USD', 'available': False, 'calories': 146, 'dietaryTags': ['High Protein', 'Gluten-Free'], 'content': 'Ashunti`Way Smoothie belongs to category Smoothies. Fruit n greens, mango bananas, tropical fruit blend, dragon fruit mix, mango, bananas, pineapples, apples, and spinach. Special green with strawberry bananas juice blend . Our fruity tasty smoothies are blended to perfection. Price is 5.49 USD.', 'metadata': {'createdBy': 'AI200-Lab', '

### Execute a Multi-Property Sort Query

In [9]:
query = """
SELECT *
FROM c
ORDER BY c.rating DESC, c.reviewCount DESC
"""

results = run_query(query)


=== Query Statistics ===
Documents Returned : 151
RU Charge          : 8.67
Duration (ms)      : 21.988
Query Results:
  - {'category': 'Sandwiches', 'name': 'Deli Turkey Sandwich', 'description': 'Whole (590 Cal.), Half (300 Cal.) Oven-roasted turkey breast raised without antibiotics, emerald greens, vine-ripened tomatoes, red onions, mayo, spicy mustard, salt and pepper on Country Rustic Sourdough. Allergens: Contains Wheat, Egg', 'price': '7.19 USD', 'id': 'food0074', 'restaurantId': 'rest-5', 'rating': 5, 'reviewCount': 241, 'priceValue': 7.19, 'currency': 'USD', 'available': True, 'calories': 556, 'dietaryTags': ['Vegetarian'], 'content': 'Deli Turkey Sandwich belongs to category Sandwiches. Whole (590 Cal.), Half (300 Cal.) Oven-roasted turkey breast raised without antibiotics, emerald greens, vine-ripened tomatoes, red onions, mayo, spicy mustard, salt and pepper on Country Rustic Sourdough. Allergens: Contains Wheat, Egg Price is 7.19 USD.', 'metadata': {'createdBy': 'AI200-La

### Create a Composite Index

In [8]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "compositeIndexes": [
        [
            {
                "path": "/rating",
                "order": "descending"
            },
            {
                "path": "/reviewCount",
                "order": "descending"
            }
        ]
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")


=== Indexing Policy Updated ===


### Re-run the Multi-Property Sort Query After Composite Index Creation

In [10]:
query = """
SELECT *
FROM c
ORDER BY c.rating DESC, c.reviewCount DESC
"""

results = run_query(query)


=== Query Statistics ===
Documents Returned : 151
RU Charge          : 8.67
Duration (ms)      : 21.936
Query Results:
  - {'category': 'Sandwiches', 'name': 'Deli Turkey Sandwich', 'description': 'Whole (590 Cal.), Half (300 Cal.) Oven-roasted turkey breast raised without antibiotics, emerald greens, vine-ripened tomatoes, red onions, mayo, spicy mustard, salt and pepper on Country Rustic Sourdough. Allergens: Contains Wheat, Egg', 'price': '7.19 USD', 'id': 'food0074', 'restaurantId': 'rest-5', 'rating': 5, 'reviewCount': 241, 'priceValue': 7.19, 'currency': 'USD', 'available': True, 'calories': 556, 'dietaryTags': ['Vegetarian'], 'content': 'Deli Turkey Sandwich belongs to category Sandwiches. Whole (590 Cal.), Half (300 Cal.) Oven-roasted turkey breast raised without antibiotics, emerald greens, vine-ripened tomatoes, red onions, mayo, spicy mustard, salt and pepper on Country Rustic Sourdough. Allergens: Contains Wheat, Egg Price is 7.19 USD.', 'metadata': {'createdBy': 'AI200-La

### Create a Filter + Sort Composite Index

In [11]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/rating/?"
        }
    ],
    "excludedPaths": [
        {
            "path": "/*"
        }
    ],
    "compositeIndexes": [
        [
            {
                "path": "/category",
                "order": "ascending"
            },
            {
                "path": "/rating",
                "order": "descending"
            }
        ]
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")


=== Indexing Policy Updated ===


### Run a Filter + Sort Query After Composite Index Creation

In [12]:
query = """
SELECT *
FROM c
WHERE c.category = "Smoothies"
ORDER BY c.rating DESC
"""

results = run_query(query)


=== Query Statistics ===
Documents Returned : 18
RU Charge          : 3.5
Duration (ms)      : 20.781
Query Results:
  - {'category': 'Smoothies', 'name': 'Tongue Teaser Smoothie', 'description': 'Tropical fruit blend, dragon fruit, pineapples, bananas, mango, apples, spinach, ginger powder. Special green blend, pineapple and ginger smoothies. Our fruity tasty smoothies are blended to perfection.', 'price': '6.49 USD', 'id': 'food0009', 'restaurantId': 'rest-10', 'rating': 5, 'reviewCount': 136, 'priceValue': 6.49, 'currency': 'USD', 'available': True, 'calories': 445, 'dietaryTags': ['Gluten-Free'], 'content': 'Tongue Teaser Smoothie belongs to category Smoothies. Tropical fruit blend, dragon fruit, pineapples, bananas, mango, apples, spinach, ginger powder. Special green blend, pineapple and ginger smoothies. Our fruity tasty smoothies are blended to perfection. Price is 6.49 USD.', 'metadata': {'createdBy': 'AI200-Lab', 'source': 'Restaurant Menu Dataset', 'lastUpdated': '2026-06-0

### Revert to the Default Indexing Policy

In [13]:
container_properties = container.read()

new_index_definition = """ {
    "indexingMode": "consistent",
    "automatic": true,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/_etag/?"
        }
    ],
    "fullTextIndexes": []
} """

database.replace_container(
    container = container_properties["id"],
    partition_key = container_properties['partitionKey'],
    indexing_policy = json.loads(new_index_definition)
)

print("\n=== Indexing Policy Updated ===")


=== Indexing Policy Updated ===
